# C. Custom Coadds

Create a custom Coadded image from a number of Visit images based on desired tract and patch

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from lsst.daf.butler import Butler
from lsst.ctrl.mpexec import SimplePipelineExecutor
from lsst.pipe.base import Pipeline
import os
import getpass

import lsst.afw.display as afwDisplay
import matplotlib.pyplot as plt

In [ ]:
my_username = getpass.getuser()

BASE = Path().resolve().parent.parent  # project root

## IF YOU CHANGE/ WORKING UNDER DIFFERENT FOLDER YOU MUST CHANGE THIS , KB_462 IS MINE
BASE = Path("/home/"+my_username+"/KB_462")
DATA = BASE / "data"
#OUTPUT = BASE / "outputs"

print("Working dir:", Path().resolve())

In [ ]:
# UPDATE FOR FUTURE RELEASES
butler = Butler("dp1", collections="LSSTComCam/DP1")
assert butler is not None

# Inputs

Input sorted CSV file from Part A, Part 3

In [ ]:
#
Input = DATA / "Key_A.csv"
#

df = pd.read_csv(Input)
ra       = df["RA"].to_numpy()
dec      = df["DEC"].to_numpy()
ObjID = df["LSST_objID"].to_numpy()

band = 'r'
tract = df["tract"].to_numpy()
patch = df["patch"].to_numpy()
# or
#tract = 5063
#patch = 45

This is the total number of coadded images we need to make is:

In [ ]:
unique_pairs = list(set(zip(tract, patch)))
p = len(unique_pairs)
print(p)

## Part 1

# Select Tract and Patch

Proceed one tract/patch combination at a time until Part 2

User must itterate through each tract and patch 

In [ ]:
## USER INPUT:

In [ ]:
unique_pairs = sorted(set(zip(tract, patch)))

tract_unique, patch_unique = zip(*unique_pairs)

tract_unique = np.array(tract_unique)
patch_unique = np.array(patch_unique)

my_filter = band

for i in range(len(unique_pairs)):
    tracts = tract_unique[i]
    patchs = patch_unique[i]

    print(tracts , patchs)


####
# CHOSE TRACT AND PATCH TO IMAGE 
# WE WANT TO GO 1 AT A TIME DUE TO LONG PROCESSING TIME

n = 0
#####


my_tract = tract_unique[n]
my_patch = patch_unique[n]

df_pairs = pd.DataFrame({'tract': tract, 'patch': patch})

XX = np.sum((tract == my_tract) & (patch == my_patch))

print("----------------------")
print("Selected Tract, Patch, Number of objects in image:")
print(my_tract,",", my_patch,",", XX)
print("Coadd number", (n+1), "Out of", p)

Proceed

In [ ]:
afwDisplay.setDefaultBackend('matplotlib')

In [ ]:
my_dataId = {'band': my_filter, 'tract': my_tract, 'patch': my_patch}
my_deepCoadd = butler.get('deep_coadd', dataId=my_dataId)

In [ ]:
deep_coadd_inputs = my_deepCoadd.getInfo().getCoaddInputs()

The number of total visit images in the tract and patch is:

In [ ]:
numIm = len(deep_coadd_inputs.visits)
print(numIm)

In [ ]:
deep_coadd_visits = deep_coadd_inputs.visits.asAstropy()

In [ ]:
deep_coadd_visits

In [ ]:
deep_coadd_visits.colnames

In [ ]:
visitTableRef = list(butler.registry.queryDatasets('visit_table'))
visitTable = butler.get(visitTableRef[0])

In [ ]:
visitTable.colnames

In [ ]:
visitTable.add_index('visitId')
deep_coadd_visits_mjds = visitTable.loc[deep_coadd_visits['id']]['expMidptMJD']
deep_coadd_visits['expMidptMJD'] = deep_coadd_visits_mjds

Dates of visit images

In [ ]:
plt.hist(deep_coadd_visits['expMidptMJD'])
plt.xlabel('MJD')
plt.ylabel('number of input exposures')

# Select Visit Inputs

Here we select the number of visit images to use in the deep coadd. Typically the number of visit images is much greater than 30 so we cap selection at 30 images with the highest pixel value. Coadds with more than 30 images have very long runtimes ~30+ minutes. Coadds with less than 30 images will use all available images. Excellent Good pixel are over 1,000,000.

In [ ]:
# CHOOSE IMAGES TO CREATE DEEP COADD

early_exposures = deep_coadd_visits
sorted_exposures = early_exposures[np.argsort(early_exposures['goodpix'])[::-1]]
n_exposures = len(sorted_exposures)

# OPTIONAL USER INPUT: 
# 30 for now
Q = 30
#

n_use = min(n_exposures, Q)

print(n_exposures," exposures available")
print("Using top ",n_use," exposures")

selected_exposures = sorted_exposures[:n_use]
early_exposure_ids = selected_exposures['id']

selected_exposures
#early_exposure_ids

# Initialize Coadd 

In [ ]:
drp_yaml_file = "$DRP_PIPE_DIR/pipelines/LSSTComCam/DRP-v2-compat.yaml"
uri = drp_yaml_file + "#makeDirectWarp,assembleDeepCoadd,makePsfMatchedWarp,selectDeepCoaddVisits"
pipeline = Pipeline.from_uri(uri)

In [ ]:
pipeline.addConfigOverride('makeDirectWarp', 'useVisitSummaryPsf', False)
pipeline.addConfigOverride('makeDirectWarp', 'useVisitSummaryPhotoCalib', False)
pipeline.addConfigOverride('makeDirectWarp', 'useVisitSummaryWcs', False)
pipeline.addConfigOverride('makeDirectWarp', 'connections.calexp_list', 'visit_image')

In [ ]:
my_username = getpass.getuser()
print(my_username)

In [ ]:
my_collection_identifier = 'custom_coadd_ecdfs'
my_outputCollection = 'u/'+my_username+'/'+my_collection_identifier
my_outputCollection

In [ ]:
my_visits_tupleString = "("+",".join(early_exposure_ids.astype(str))+")"
query_string = """tract = {} AND patch = {} AND visit IN {} AND skymap = 'lsst_cells_v1'
                """.format(my_tract, my_patch, my_visits_tupleString)
print(query_string)

In [ ]:
executor = SimplePipelineExecutor.from_pipeline(pipeline, butler=butler,output=my_outputCollection, where=query_string)

In [ ]:
local_repo_path = Path(f"/home/{my_username}/my_local_repo")
local_repo_path.mkdir(parents=True, exist_ok=True)
out_butler = executor.use_local_butler(local_repo_path)

# Create Coadd

Expected runtime is ~25 minutes for 30 visit images 

In [ ]:
executor.run(register_dataset_types=True)

In [ ]:
deepCoadd = out_butler.get('deep_coadd_predetection', tract=my_tract, patch=my_patch,
                           band='r', instrument='LSSTComCam', skymap='lsst_cells_v1',
                           collections=executor.quantum_graph.metadata["output_run"])

# View Coadd

Display and save coadd image

In [ ]:
fig = plt.figure(figsize=(10, 8))
afw_display = afwDisplay.Display(frame=fig)
afw_display.scale('asinh', 'zscale')
afw_display.mtv(deepCoadd.image)
plt.gca().axis('on')

# Toggle to save figs
COADD_PATH = DATA / "plots" / "coadds"
COADD_PATH.mkdir(parents=True, exist_ok=True)
plt.savefig(COADD_PATH / f"{my_tract}_{my_patch}_Coadd.png",
            dpi=300, bbox_inches="tight")

plt.show()
plt.close()  #

# Store .fits File

In [ ]:
refs = out_butler.query_datasets('deep_coadd_predetection',
                                 collections=executor.quantum_graph.metadata["output_run"])


# --- DEFINE OUTPUT DIRECTORY ---
FITS_PATH = DATA / "fits_files"
FITS_PATH.mkdir(parents=True, exist_ok=True)

# --- RETRIEVE FILES ---
Name = out_butler.retrieveArtifacts(
    refs,
    FITS_PATH,
    preserve_path=False,
    transfer="copy",
    overwrite=True
)

print("Saved files:")
for f in Name:
    print(f)

Verify image has been added to home directory 

In [ ]:
print(os.listdir(FITS_PATH))

# Part 2

# Append Progress CSV

Use this CSV to track progress as this step takes a long time

In [ ]:
import pandas as pd
import os

filename = DATA/ "Tract_Patch_Visit_Name.csv"

row_df = pd.DataFrame([{
    "tract": my_tract,
    "patch": my_patch,
    "NumVisIm": n_use,
    "Name": Name[0],
    "NumObjects" : XX # or str(Name[0])
}])

row_df.to_csv(
    filename,
    mode='a',
    header=not os.path.exists(filename),
    index=False
)

You have sucessfully created and saved a custom deep coadded image, from here either repeat Part A for the next tract/patch comination, n, or proceed to Part 4

In [ ]:
# EITHER PROCEED WITH MAKING ALL COADDS, THEN TO "Coadd_View.ipynb",  OR

In [ ]:
# PROCEED WITH ONE COADD TO "Coadd_View.ipynb"

In [ ]:
# OPTIONAL: CLEAN UP DIRECTORY

In [ ]:
#os.remove("")